In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
rawdata=pd.read_csv('realtor-data.zip.csv')
np.random.seed(0)

In [17]:
total_empty=rawdata.isnull().sum()
print((total_empty))


brokered_by         4533
status                 0
price               1541
bed               481317
bath              511771
acre_lot          325589
street             10866
city                1407
state                  8
zip_code             299
house_size        568484
prev_sold_date    734297
dtype: int64


Here we can see that prev_sold_date column has the highest number of missing values
now let us see the percentage of the missing values in the data overall and per column


In [18]:
total_cells=rawdata.shape[0]*rawdata.shape[1]
total_nans=total_empty.sum()
total_percentage_empty=total_nans/total_cells*100
print(total_percentage_empty)

9.88192203015176


Above we can see that the total empty percentage isn't much
now we will see per column how much data is missing.

In [19]:
total_rows=len(rawdata)
missing_percentage_column=total_empty/total_rows*100
missing_info=pd.DataFrame({'missing count':total_empty,'missing percentage':missing_percentage_column}).sort_values(by='missing percentage',ascending=False)
print(missing_info)


                missing count  missing percentage
prev_sold_date         734297           32.981627
house_size             568484           25.533983
bath                   511771           22.986666
bed                    481317           21.618797
acre_lot               325589           14.624130
street                  10866            0.488056
brokered_by              4533            0.203604
price                    1541            0.069215
city                     1407            0.063197
zip_code                  299            0.013430
state                       8            0.000359
status                      0            0.000000


Since the missing percentage of the columns below acre_lot are below 1% it will not make a huge difference in deleteing those rows


In [27]:
low_missing_cols=list(missing_info[missing_info['missing percentage']<=1].index)
print(low_missing_cols)



['street', 'brokered_by', 'price', 'city', 'zip_code', 'state', 'status']


In [32]:
original_row_count=len(rawdata)
cleaned_data=rawdata.dropna(subset=low_missing_cols)
dropped_rows=original_row_count-len(cleaned_data)
print(f'the original row count = {original_row_count}')
print(f'after removing the unwanted rows = {len(cleaned_data)}')
print(f'total rows dropped = {dropped_rows}')
cleaned_data.head()

the original row count = 2226382
after removing the unwanted rows = 2207981
total rows dropped = 18401


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


Since the data in the prev_sold_date is empty , it might be mostly due to the houses being new and never sold .
so therefore, i will replace the empty values as 'never_sold'. we cant delete the entire column since the missing percentage is very high and deleting the column will also cause issue while predicting the price of the house.


In [34]:
cleaned_data['prev_sold_date']=cleaned_data['prev_sold_date'].fillna('never_sold')
(cleaned_data.head())

C:\Users\sajja\AppData\Local\Temp\ipykernel_6184\1135527588.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_data['prev_sold_date']=cleaned_data['prev_sold_date'].fillna('never_sold')


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,never_sold
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,never_sold
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,never_sold
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,never_sold
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,never_sold


now we need to check the remaining empty values in other columns and start dealing with them


In [36]:
leftover_empty=cleaned_data.isnull().sum()
print(leftover_empty)

brokered_by            0
status                 0
price                  0
bed               474205
bath              503566
acre_lot          321631
street                 0
city                   0
state                  0
zip_code               0
house_size        560807
prev_sold_date         0
dtype: int64


there are 4 columns which still have empty values . Now I will use some variety of imputation methods to solve these missing data
For the acre_lot missing values , i can use the zip_code (since there are no missing values in it) and use the group-wise imputation method here. 


In [51]:
cleaned_data['acre_lot']=cleaned_data.groupby('zip_code')['acre_lot'].transform(lambda x:x.fillna(x.median()))
cleaned_data.head()

C:\Users\sajja\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\sajja\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\sajja\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\sajja\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarnin

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,never_sold
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,never_sold
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,never_sold
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,never_sold
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,never_sold


we should have now solved the missing values in the acre_lot column. Lets quickly verify again and go onto clean the remaining other column.

In [53]:
total_leftover_nans=cleaned_data.isnull().sum()
print(total_leftover_nans)

brokered_by            0
status                 0
price                  0
bed               474205
bath              503566
acre_lot            1345
street                 0
city                   0
state                  0
zip_code               0
house_size        560807
prev_sold_date         0
dtype: int64


Oops, seems like there is still some more empty places left . This is most likely due to the entire group of those zip_code being empty.
for this , i will now use the global median to fill the empty places , but i will also flag these empty nans before in case there might ever be some connections with these empty acre_lot values.


In [55]:
cleaned_data['acre_lot_IMPUTED']=cleaned_data['acre_lot'].isnull().astype(int)
median_acre_lot=cleaned_data['acre_lot'].median()
cleaned_data['acre_lot'].fillna(median_acre_lot)

0          0.12
1          0.08
2          0.15
3          0.10
4          0.05
           ... 
2226377    0.33
2226378    0.10
2226379    0.50
2226380    0.09
2226381    0.31
Name: acre_lot, Length: 2207981, dtype: float64

Now We should be clear of any empty values in the acre_lot column

In [58]:
cleaned_data.head()
total_leftover_nans=cleaned_data.isnull().sum()
print(total_leftover_nans)


brokered_by              0
status                   0
price                    0
bed                 474205
bath                503566
acre_lot                 0
street                   0
city                     0
state                    0
zip_code                 0
house_size          560807
prev_sold_date           0
acre_lot_IMPUTED         0
dtype: int64


Now for the bed and bath empty values, we will use mode since the number of bed and bath are dependent on the imediated neighborhood.But still i will flag these empty value in case they might hold any significance.



In [ ]:
cleaned_data['bed_IMPUTED']=cleaned_data['bed'].isnull().astype(int)
cleaned_data['bath_IMPUTED']=cleaned_data['bath'].isnull().astype(int)
cleaned_data['bed']=cleaned_data.groupby('zip_code')['bed'].transform(lambda x:x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
cleaned_data['bath']=cleaned_data.groupby('zip_code')['bath'].transform(lambda x:x.fillna(x.mode()[0] if not x.mode().empty else np.nan))


As we can see the same issue which had ran with acre_lot column has occured so therefore , we will use the same solution for the same problem :)

In [78]:
global_bed_mode=cleaned_data['bed'].mode()[0]
global_bath_mode=cleaned_data['bath'].mode()[0]
cleaned_data['bed'].fillna(global_bed_mode,inplace=True)
cleaned_data['bath'].fillna(global_bath_mode,inplace=True)


C:\Users\sajja\AppData\Local\Temp\ipykernel_6184\983554249.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cleaned_data['bed'].fillna(global_bed_mode,inplace=True)
C:\Users\sajja\AppData\Local\Temp\ipykernel_6184\983554249.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

In [79]:
print(cleaned_data[['bed','bath']].isnull().sum())

bed     0
bath    0
dtype: int64


Now that we have fixed all these two columns , its time to finish the final column the 'house_size'  !!!

In [80]:
print(cleaned_data.isnull().sum())

brokered_by              0
status                   0
price                    0
bed                      0
bath                     0
acre_lot                 0
street                   0
city                     0
state                    0
zip_code                 0
house_size          560807
prev_sold_date           0
acre_lot_IMPUTED         0
bed_IMPUTED              0
bath_IMPUTED             0
dtype: int64
